# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [1]:
from class_dataset import SegmentationClassDataset, num_classes as class_num_classes
from instance_dataset import SegmentationInstanceDataset

# mode = "instance"
mode = "class"

if mode == "instance":
    dataset_class = SegmentationInstanceDataset
    num_classes = 2
    model_path = "model_instance.pth"
elif mode == "class":
    dataset_class = SegmentationClassDataset
    num_classes = class_num_classes
    model_path = "model_class.pth"

In [2]:
from dataset_transforms import get_transforms
from model import get_model_instance_segmentation
import utils

import torch


model = get_model_instance_segmentation(num_classes)

dataset = dataset_class("../../training_data", get_transforms(is_train=True))
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

{'loss_classifier': tensor(1.7568, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.5098, grad_fn=<DivBackward0>), 'loss_mask': tensor(1.0039, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_objectness': tensor(2.2678, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_rpn_box_reg': tensor(0.3787, grad_fn=<DivBackward0>)}
{'boxes': tensor([[4.7704e+01, 1.2409e+01, 2.1728e+02, 3.0000e+02],
        [3.2090e+02, 2.2033e+02, 4.0000e+02, 2.9374e+02],
        [2.0500e+02, 1.4918e+01, 3.5451e+02, 2.9617e+02],
        [1.6279e+02, 2.9817e+02, 3.4785e+02, 2.9991e+02],
        [0.0000e+00, 2.9768e+02, 1.5411e+02, 2.9990e+02],
        [1.0237e+02, 5.9267e+00, 2.7453e+02, 3.0000e+02],
        [3.5500e+02, 2.9608e+02, 4.0000e+02, 2.9978e+02],
        [3.4459e+02, 1.6387e-01, 4.0000e+02, 2.8920e+02],
        [3.6109e+02, 2.8440e+02, 4.0000e+02, 2.9913e+02],
        [2.9736e+02, 1.5059e+02, 4.0000e+02, 2.9151e+02],
        [3.6143e+02, 1.7451e+02, 4.0000e+02, 2.9417e+02],
        [3

## Training and Validation


In [3]:
import csv
from engine import evaluate, train_one_epoch
from sklearn.model_selection import KFold


k_folds = 5
num_epochs = 6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training.")

kfold = KFold(n_splits=k_folds, shuffle=True)

# use our dataset and defined transformations
full_dataset = dataset_class("../../training_data", get_transforms(is_train=True))

results = [
    [
        "fold",
        "epoch",
        "batch",
        "loss_classifier",
        "loss_box_reg",
        "loss_mask",
        "loss_objectness",
        "loss_rpn_box_reg",
    ]
]

for fold, (train_ids, test_ids) in enumerate(kfold.split(full_dataset)):
    print(f"FOLD {fold}")
    print("-" * 20)

    train_dataset = torch.utils.data.Subset(full_dataset, train_ids)
    train_dataset.dataset.transforms = get_transforms(is_train=True)

    valid_dataset = torch.utils.data.Subset(full_dataset, test_ids)
    valid_dataset.dataset.transforms = get_transforms(is_train=False)

    # define training and validation data loaders
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=1, shuffle=True, collate_fn=utils.collate_fn
    )
    validation_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=1, shuffle=False, collate_fn=utils.collate_fn
    )

    model = get_model_instance_segmentation(num_classes).to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    for epoch in range(num_epochs):
        # train for one epoch, printing every 10 iterations
        training_losses = train_one_epoch(
            model, optimizer, train_loader, device, epoch, print_freq=10
        )

        for batch, losses in enumerate(training_losses):
            results.append(
                [
                    fold,
                    epoch,
                    batch,
                    losses["loss_classifier"],
                    losses["loss_box_reg"],
                    losses["loss_mask"],
                    losses["loss_objectness"],
                    losses["loss_rpn_box_reg"],
                ]
            )

        # update the learning rate
        lr_scheduler.step()
        # evaluate on the validation dataset
        evaluate(model, validation_loader, device=device)

# save results to a CSV file
with open("results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

Using cpu for training.
FOLD 0
--------------------
Epoch: [0]  [ 0/96]  eta: 0:08:16  lr: 0.000058  loss: 5.3236 (5.3236)  loss_classifier: 1.7441 (1.7441)  loss_box_reg: 0.3724 (0.3724)  loss_mask: 0.8979 (0.8979)  loss_objectness: 1.5780 (1.5780)  loss_rpn_box_reg: 0.7313 (0.7313)  time: 5.1768  data: 0.3865
Epoch: [0]  [10/96]  eta: 0:07:13  lr: 0.000583  loss: 4.3023 (4.2908)  loss_classifier: 1.6304 (1.4766)  loss_box_reg: 0.3985 (0.3922)  loss_mask: 0.8593 (0.7859)  loss_objectness: 1.2808 (1.2443)  loss_rpn_box_reg: 0.2297 (0.3917)  time: 5.0391  data: 0.5071
Epoch: [0]  [20/96]  eta: 0:06:19  lr: 0.001109  loss: 2.4243 (3.1288)  loss_classifier: 0.8659 (1.0299)  loss_box_reg: 0.4222 (0.3908)  loss_mask: 0.5782 (0.6254)  loss_objectness: 0.2971 (0.8235)  loss_rpn_box_reg: 0.1468 (0.2592)  time: 4.9774  data: 0.4941
Epoch: [0]  [30/96]  eta: 0:05:27  lr: 0.001635  loss: 1.4139 (2.5324)  loss_classifier: 0.4044 (0.7905)  loss_box_reg: 0.3960 (0.3942)  loss_mask: 0.3869 (0.5163)  

KeyboardInterrupt: 

In [ ]:
# Train final model on all data

print(f"FINAL TRAINING")
print("-" * 20)

# define training and validation data loaders
train_loader = torch.utils.data.DataLoader(
    full_dataset, batch_size=1, shuffle=True, collate_fn=utils.collate_fn
)

model = get_model_instance_segmentation(num_classes).to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

results = [
    [
        "epoch",
        "batch",
        "loss_classifier",
        "loss_box_reg",
        "loss_mask",
        "loss_objectness",
        "loss_rpn_box_reg",
    ]
]

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    training_losses = train_one_epoch(
        model, optimizer, train_loader, device, epoch, print_freq=10
    )

    for batch, losses in enumerate(training_losses):
        results.append(
            [
                epoch,
                batch,
                losses["loss_classifier"],
                losses["loss_box_reg"],
                losses["loss_mask"],
                losses["loss_objectness"],
                losses["loss_rpn_box_reg"],
            ]
        )

    # update the learning rate
    lr_scheduler.step()

# save results to a CSV file
with open("final-results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

# save the model
torch.save(model.state_dict(), model_path)